# Chronos-2 FD001 — Multi-seed evaluation + statistical tests (MCCSAI revision)

Same pipeline as W06 (FD001, global MinMax on 24 variables, W=30, RUL cap 125, last window per test unit).

**What this notebook produces for the revised paper**
1. Dataset facts (cycles, windows, shortest test trajectory → justifies W=30, feature counts).
2. Chronos-2 × {mean, last, concat} × {Ridge, MLP, XGBoost} × {correlation, AFICv} over **5 seeds** → mean ± std.
3. Re-trained baselines under the *identical* protocol, 5 seeds: XGBoost on raw windows, task-specific LSTM.
4. Optional: Chronos-1 + heads, 5 seeds.
5. Statistics: bootstrap 95% CI of RMSE over the 100 test engines + paired Wilcoxon tests on per-engine errors.

Embeddings are deterministic → extracted **once** (one forward pass gives all 3 poolings) and cached on Drive.
Seeds: 42, 123, 456, 789, 1000. **Runtime: GPU (T4).**

**Resumable:** embeddings are saved chunk by chunk and every (configuration, seed) result is saved to Drive
as soon as it is computed. If Colab disconnects, reconnect and simply **Run all** again: finished work is
reloaded, only what is missing is computed.


In [ ]:
!pip install -q 'chronos-forecasting>=2.2' xgboost torch

import os
os.chdir('/content')
!rm -rf /content/RUL-Chronos
!git clone https://github.com/f-khadija-benzine/RUL-Chronos.git /content/RUL-Chronos
os.chdir('/content/RUL-Chronos/src')

from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, time, warnings
warnings.filterwarnings('ignore')
import torch
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from scipy.stats import wilcoxon

from data_loader import MultiDatasetLoader
from preprocessing import DataNormalizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR  = '/content/drive/MyDrive/PhD_results/offline/FD001_multiseed'
EMB_ROOT = '/content/drive/MyDrive/PhD_results/embeddings_cache/FD001/chronos2_pool'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(EMB_ROOT, exist_ok=True)
SEEDS = [42, 123, 456, 789, 1000]
print('Device:', DEVICE)

## A — Data (identical to W06)

In [ ]:
DATASET, W, RUL_CAP = 'FD001', 30, 125
loader = MultiDatasetLoader()
ds = loader.load_cmapss_dataset(DATASET)
train_raw, test_raw = ds['train'].copy(), ds['test'].copy()
train_raw['rul'] = train_raw['rul'].clip(upper=RUL_CAP)
if 'rul' in test_raw.columns:
    test_raw['rul'] = test_raw['rul'].clip(upper=RUL_CAP)

sensor_cols  = [c for c in train_raw.columns if c.startswith('sensor_')]
setting_cols = [c for c in train_raw.columns if c.startswith('setting_')]
feature_cols = sensor_cols + setting_cols

norm = DataNormalizer(method='minmax')
train_norm = norm.fit_transform(train_raw, feature_cols)
test_norm  = norm.transform(test_raw)

# ---- Dataset facts for the paper (Reviewer 1: dataset size, W=30) ----
tr_len = train_raw.groupby('unit')['cycle'].max()
te_len = test_raw.groupby('unit')['cycle'].max()
FACTS = {
  'train_units': int(train_raw['unit'].nunique()), 'test_units': int(test_raw['unit'].nunique()),
  'train_cycles': int(len(train_raw)), 'test_cycles': int(len(test_raw)),
  'raw_variables': len(feature_cols),
  'train_traj_min_mean_max': [int(tr_len.min()), float(round(tr_len.mean(),1)), int(tr_len.max())],
  'test_traj_min_mean_max':  [int(te_len.min()), float(round(te_len.mean(),1)), int(te_len.max())],
  'test_units_shorter_than_W': int((te_len < W).sum()),
}
for k,v in FACTS.items(): print(f'{k:28s} {v}')

In [ ]:
def variance_filter(df, cols, threshold=0.001):
    v = df[cols].var(); sel = v[v > threshold].index.tolist()
    return sel, [c for c in cols if c not in sel]

def correlation_filter(df, cols, threshold=0.95):
    cm = df[cols].corr().abs()
    up = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))
    drop = [c for c in up.columns if any(up[c] > threshold)]
    return [c for c in cols if c not in drop], drop

def aficv_selection(df, cols, target_col='rul', threshold=0.90):
    m = XGBRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    m.fit(df[cols].values, df[target_col].values)
    imp = m.feature_importances_; idx = np.argsort(imp)[::-1]
    cs = np.cumsum(imp[idx]); n = np.searchsorted(cs, threshold) + 1
    return [cols[i] for i in idx[:n]], cs[n-1]

feats_var, rm_var   = variance_filter(train_norm, feature_cols)
feats_corr, rm_corr = correlation_filter(train_norm, feats_var)
feats_aficv, cov    = aficv_selection(train_norm, feature_cols)
feature_sets = {'correlation': feats_corr, 'aficv': feats_aficv}
FACTS.update({'n_after_variance': len(feats_var), 'removed_variance': rm_var,
              'n_correlation': len(feats_corr), 'removed_correlation': rm_corr,
              'n_aficv': len(feats_aficv), 'aficv_features': feats_aficv, 'aficv_coverage': float(cov)})
print(f'24 -> variance {len(feats_var)} -> correlation {len(feats_corr)} | AFICv {len(feats_aficv)} ({cov:.1%})')

In [ ]:
def create_sliding_windows(df, cols, window_size=30):
    X, y, U = [], [], []
    for u in sorted(df['unit'].unique()):
        d = df[df['unit'] == u].sort_values('cycle')
        if len(d) < window_size: continue
        v, r = d[cols].values, d['rul'].values
        for s in range(len(d) - window_size + 1):
            X.append(v[s:s+window_size]); y.append(r[s+window_size-1]); U.append(u)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), np.array(U)

data_dict = {}
for fs, cols in feature_sets.items():
    Xtr, ytr, utr = create_sliding_windows(train_norm, cols, W)
    Xte, yte, ute = create_sliding_windows(test_norm,  cols, W)
    data_dict[fs] = dict(X_train=Xtr, y_train=ytr, units_train=utr, X_test=Xte, y_test=yte, units_test=ute)
    print(f'{fs}: X_train={Xtr.shape}  X_test={Xte.shape}  test units evaluated={len(np.unique(ute))}')
FACTS['train_windows'] = int(len(data_dict['correlation']['y_train']))
FACTS['test_windows']  = int(len(data_dict['correlation']['y_test']))
json.dump(FACTS, open(f'{OUT_DIR}/fd001_facts.json','w'), indent=2, default=str)

In [ ]:
def nasa_score(y_true, y_pred):
    d = y_pred - y_true
    return float(np.sum(np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1)))

def last_per_unit(y_true, y_pred, units):
    us = sorted(set(units))
    t = np.array([y_true[units == u][-1] for u in us])
    p = np.array([y_pred[units == u][-1] for u in us])
    return t, p

def metrics(t, p):
    return float(np.sqrt(mean_squared_error(t, p))), nasa_score(t, p)

_d = data_dict['correlation']
Y_TRUE = last_per_unit(_d['y_test'], _d['y_test'], _d['units_test'])[0]

# ---------- RESUMABLE RUN CACHE ----------
# Every (configuration, seed) result is saved on Drive as soon as it is computed.
# If the runtime disconnects, just re-run all cells: finished runs are loaded, not recomputed.
RUN_DIR = f'{OUT_DIR}/runs'
os.makedirs(RUN_DIR, exist_ok=True)

def cached_run(name, seed, fn):
    path = f"{RUN_DIR}/{name.replace('|', '__')}__seed{seed}.npy"
    if os.path.exists(path):
        return np.load(path)
    pred = np.asarray(fn(), dtype=np.float32)     # per-engine predictions (100,)
    np.save(path, pred)
    print(f'    saved {os.path.basename(path)}  RMSE={metrics(Y_TRUE, pred)[0]:.2f}')
    return pred

PRED, ROWS = {}, []

def record(name, fs, pooling, head, dim, preds_per_seed):
    t = Y_TRUE
    P = np.vstack(preds_per_seed); PRED[name] = P
    r = np.array([metrics(t, p)[0] for p in P]); s = np.array([metrics(t, p)[1] for p in P])
    sd = lambda a: a.std(ddof=1) if len(a) > 1 else 0.0
    ROWS.append(dict(Config=name, Feature_Set=fs, Pooling=pooling, Head=head, Dim=dim,
               RMSE_mean=r.mean(), RMSE_std=sd(r), RMSE_min=r.min(), RMSE_max=r.max(),
               Score_mean=s.mean(), Score_std=sd(s), n_seeds=len(P)))
    print(f'  {name:45s} RMSE {r.mean():.2f} ± {sd(r):.2f}  [min {r.min():.2f}, max {r.max():.2f}]'
          f'   Score {s.mean():.1f} ± {sd(s):.1f}')
    pd.DataFrame(ROWS).to_csv(f'{OUT_DIR}/fd001_multiseed_all.csv', index=False)
print('helpers + resumable run cache ✓  (', len(os.listdir(RUN_DIR)), 'runs already on Drive )')

## B — Chronos-2 embeddings (one pass → mean / last / concat, cached)

In [ ]:
from chronos import Chronos2Pipeline
chronos2 = Chronos2Pipeline.from_pretrained('amazon/chronos-2', device_map=DEVICE)
try:
    n_params = sum(p.numel() for p in chronos2.model.parameters())
    FACTS['chronos2_params_M'] = round(n_params/1e6, 1); print(f'amazon/chronos-2 loaded — {n_params/1e6:.1f}M parameters')
except Exception as e:
    print('amazon/chronos-2 loaded (param count unavailable:', e, ')')
json.dump(FACTS, open(f'{OUT_DIR}/fd001_facts.json','w'), indent=2, default=str)

POOLINGS = ['mean', 'last', 'concat']
CHUNK = 2048   # windows per checkpoint (~1-2 min on a T4)

def _pool(emb):
    return {'mean':   emb.mean(dim=(1, 2)).cpu().numpy(),
            'last':   emb[:, :, -1, :].mean(dim=1).cpu().numpy(),
            'concat': emb.mean(dim=1).reshape(emb.shape[0], -1).cpu().numpy()}

def extract_chunk(X, batch_size=64):
    out = {p: [] for p in POOLINGS}
    for i in range(0, len(X), batch_size):
        bt = torch.tensor(np.transpose(X[i:i+batch_size], (0, 2, 1)), dtype=torch.float32)  # keep on CPU
        with torch.no_grad():
            emb = torch.stack(chronos2.embed(bt)[0]).float()     # (B, F, P, 768)
        for p, v in _pool(emb).items(): out[p].append(v)
    return {p: np.vstack(v).astype(np.float32) for p, v in out.items()}

def get_split(fs, split, X):
    # Resumable: each chunk is saved to Drive; an interrupted run restarts at the first missing chunk.
    final = {p: f'{EMB_ROOT}/{fs}_{p}_{split}.npy' for p in POOLINGS}
    if all(os.path.exists(v) for v in final.values()):
        return {p: np.load(v) for p, v in final.items()}
    n_chunks = int(np.ceil(len(X) / CHUNK))
    chunk_dir = f'{EMB_ROOT}/chunks_{fs}_{split}'; os.makedirs(chunk_dir, exist_ok=True)
    for k in range(n_chunks):
        f = f'{chunk_dir}/chunk{k:03d}.npz'
        if os.path.exists(f): continue
        t0 = time.time()
        np.savez(f, **extract_chunk(X[k*CHUNK:(k+1)*CHUNK]))
        print(f'  [{fs}|{split}] chunk {k+1}/{n_chunks} saved ({time.time()-t0:.0f}s)')
    parts = [np.load(f'{chunk_dir}/chunk{k:03d}.npz') for k in range(n_chunks)]
    res = {p: np.vstack([c[p] for c in parts]) for p in POOLINGS}
    for p in POOLINGS: np.save(final[p], res[p])
    return res

def get_emb(fs):
    d = data_dict[fs]
    tr = get_split(fs, 'train', d['X_train']); te = get_split(fs, 'test', d['X_test'])
    return {p: (tr[p], te[p]) for p in POOLINGS}

EMB = {fs: get_emb(fs) for fs in feature_sets}
for fs in EMB:
    for p in POOLINGS: print(fs, p, EMB[fs][p][0].shape)

## C — Heads × 5 seeds

In [ ]:
def make_heads(seed):
    return [('Ridge',   lambda: Ridge(alpha=1.0)),
            ('MLP',     lambda: MLPRegressor(hidden_layer_sizes=(256,128), max_iter=500, early_stopping=True,
                                             n_iter_no_change=10, random_state=seed)),
            ('XGBoost', lambda: XGBRegressor(n_estimators=100, max_depth=5, random_state=seed, n_jobs=-1))]

def fit_predict(make_model, Xtr, ytr, Xte, units_te, y_te):
    def _f():
        m = make_model(); m.fit(Xtr, ytr)
        return last_per_unit(y_te, m.predict(Xte), units_te)[1]
    return _f

for fs in feature_sets:
    d = data_dict[fs]
    for p in POOLINGS:
        Xtr, Xte = EMB[fs][p]
        print(f'\n=== Chronos-2 | {fs} | {p} (dim {Xtr.shape[1]}) ===')
        for h in ['Ridge', 'MLP', 'XGBoost']:
            name = f'C2|{fs}|{p}|{h}'
            seeds = [0] if h == 'Ridge' else SEEDS          # Ridge is deterministic
            preds = [cached_run(name, sd, fit_predict(dict(make_heads(sd))[h], Xtr, d['y_train'],
                                                          Xte, d['units_test'], d['y_test']))
                     for sd in seeds]
            record(name, fs, p, h, Xtr.shape[1], preds)
pd.DataFrame(ROWS).to_csv(f'{OUT_DIR}/fd001_multiseed_all.csv', index=False)

## D — Baselines re-trained under the identical protocol (correlation features, 5 seeds)

In [ ]:
d = data_dict['correlation']
Xtr_flat = d['X_train'].reshape(len(d['X_train']), -1); Xte_flat = d['X_test'].reshape(len(d['X_test']), -1)

# D1. XGBoost directly on raw windows (no foundation model)
preds = [cached_run('BASE|XGBoost-raw', sd,
          fit_predict(lambda sd=sd: XGBRegressor(n_estimators=100, max_depth=5, random_state=sd, n_jobs=-1),
                      Xtr_flat, d['y_train'], Xte_flat, d['units_test'], d['y_test']))
         for sd in SEEDS]
record('BASE|XGBoost-raw', 'correlation', '-', 'XGBoost', Xtr_flat.shape[1], preds)

# D2. Task-specific LSTM trained end-to-end
import torch.nn as nn
class LSTMReg(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.lstm = nn.LSTM(n_in, 64, num_layers=2, batch_first=True, dropout=0.2)
        self.head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x): return self.head(self.lstm(x)[0][:, -1]).squeeze(-1)

def train_lstm(seed, epochs=60, bs=256, patience=10):
    torch.manual_seed(seed); np.random.seed(seed)
    # validation = 10% of TRAIN engines (never the test set)
    units = np.unique(d['units_train']); rng = np.random.RandomState(seed)
    val_u = rng.choice(units, size=max(1, len(units)//10), replace=False)
    vm = np.isin(d['units_train'], val_u)
    Xt, yt = torch.tensor(d['X_train'][~vm]), torch.tensor(d['y_train'][~vm])
    Xv, yv = torch.tensor(d['X_train'][vm]).to(DEVICE), torch.tensor(d['y_train'][vm]).to(DEVICE)
    model = LSTMReg(Xt.shape[2]).to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    best, best_state, wait = 1e9, None, 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            idx = perm[i:i+bs]; xb, yb = Xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            opt.zero_grad(); loss = ((model(xb) - yb)**2).mean(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): vl = ((model(Xv) - yv)**2).mean().item()
        if vl < best: best, best_state, wait = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad(): yp = model(torch.tensor(d['X_test']).to(DEVICE)).cpu().numpy()
    return last_per_unit(d['y_test'], yp, d['units_test'])

preds = [cached_run('BASE|LSTM', sd, lambda sd=sd: train_lstm(sd)[1]) for sd in SEEDS]
record('BASE|LSTM', 'correlation', '-', 'LSTM', 0, preds)

## E — (Optional, slow ~CPU) Chronos-1 + heads, 5 seeds — run if time allows

In [ ]:
RUN_CHRONOS1 = True   # set False if short on time
if RUN_CHRONOS1:
    from chronos import ChronosPipeline
    c1 = ChronosPipeline.from_pretrained('amazon/chronos-t5-small', device_map='cpu', torch_dtype=torch.float32)
    def c1_embed_chunk(X, bs=256):
        out = []
        for i in range(0, len(X), bs):
            b = X[i:i+bs]; per = []
            for f in range(b.shape[2]):
                with torch.no_grad():
                    e = c1.embed(torch.tensor(b[:, :, f]))[0]
                per.append(e.mean(dim=1).float().numpy())
            out.append(np.concatenate(per, axis=1))
        return np.vstack(out).astype(np.float32)
    def c1_split(split, X):
        final = f'{EMB_ROOT}/c1_correlation_{split}.npy'
        if os.path.exists(final): return np.load(final)
        cdir = f'{EMB_ROOT}/chunks_c1_{split}'; os.makedirs(cdir, exist_ok=True)
        n = int(np.ceil(len(X) / CHUNK))
        for k in range(n):
            f = f'{cdir}/chunk{k:03d}.npy'
            if os.path.exists(f): continue
            t0 = time.time(); np.save(f, c1_embed_chunk(X[k*CHUNK:(k+1)*CHUNK]))
            print(f'  [Chronos-1|{split}] chunk {k+1}/{n} saved ({time.time()-t0:.0f}s)')
        E = np.vstack([np.load(f'{cdir}/chunk{k:03d}.npy') for k in range(n)]); np.save(final, E)
        return E
    E1tr, E1te = c1_split('train', d['X_train']), c1_split('test', d['X_test'])
    for h in ['Ridge', 'MLP', 'XGBoost']:
        name = f'C1|correlation|mean|{h}'
        seeds = [0] if h == 'Ridge' else SEEDS
        preds = [cached_run(name, sd, fit_predict(dict(make_heads(sd))[h], E1tr, d['y_train'],
                                                    E1te, d['units_test'], d['y_test'])) for sd in seeds]
        record(name, 'correlation', 'mean', h, E1tr.shape[1], preds)
    pd.DataFrame(ROWS).to_csv(f'{OUT_DIR}/fd001_multiseed_all.csv', index=False)

## F — Results table + statistics

In [ ]:
res = pd.DataFrame(ROWS).sort_values('RMSE_mean')
res['RMSE'] = res.apply(lambda r: f"{r.RMSE_mean:.2f} ± {r.RMSE_std:.2f}", axis=1)
res['Score'] = res.apply(lambda r: f"{r.Score_mean:.1f} ± {r.Score_std:.1f}", axis=1)
print(res[['Config','Dim','RMSE','RMSE_min','RMSE_max','Score']].to_string(index=False))
res.to_csv(f'{OUT_DIR}/fd001_multiseed_all.csv', index=False)

In [ ]:
# Bootstrap 95% CI of RMSE over the 100 test engines (seed-averaged predictions)
# + paired Wilcoxon signed-rank test on per-engine absolute errors vs the best Chronos-2 config.
rng = np.random.RandomState(0); B = 5000
def boot_ci(p, t):
    idx = rng.randint(0, len(t), size=(B, len(t)))
    r = np.sqrt(((p[idx] - t[idx])**2).mean(axis=1)); return np.percentile(r, [2.5, 97.5])

best_c2 = res[res.Config.str.startswith('C2|correlation')].iloc[0].Config
ref = PRED[best_c2].mean(axis=0); t = Y_TRUE
stat_rows = []
for name in res.Config:
    p = PRED[name].mean(axis=0)
    lo, hi = boot_ci(p, t)
    # paired bootstrap of the RMSE difference (name - best_c2)
    idx = rng.randint(0, len(t), size=(B, len(t)))
    diff = np.sqrt(((p[idx]-t[idx])**2).mean(1)) - np.sqrt(((ref[idx]-t[idx])**2).mean(1))
    dlo, dhi = np.percentile(diff, [2.5, 97.5])
    pw = np.nan if name == best_c2 else wilcoxon(np.abs(p - t), np.abs(ref - t)).pvalue
    stat_rows.append(dict(Config=name, RMSE_ensemble=np.sqrt(((p-t)**2).mean()), CI95_lo=lo, CI95_hi=hi,
                          dRMSE_vs_best=np.sqrt(((p-t)**2).mean()) - np.sqrt(((ref-t)**2).mean()),
                          dCI_lo=dlo, dCI_hi=dhi, wilcoxon_p=pw))
stats = pd.DataFrame(stat_rows).round(3)
print('Reference (best Chronos-2):', best_c2)
print(stats.to_string(index=False))
stats.to_csv(f'{OUT_DIR}/fd001_stats.csv', index=False)
np.savez(f'{OUT_DIR}/fd001_unit_preds.npz', y_true=Y_TRUE, **{k.replace('|','__'): v for k, v in PRED.items()})
print('\nSaved to', OUT_DIR)

In [ ]:
# ---- Paste this whole output back to Claude ----
print(json.dumps(FACTS, indent=1, default=str))
print(res[['Config','Dim','RMSE','RMSE_min','RMSE_max','Score']].to_string(index=False))
print(stats.to_string(index=False))